# 2. etapa: doladenie transformera (E1, E2, E1b)

Notebook je určený na Google Colab s GPU (*Runtime → Change runtime type → GPU*). Spúšťa skript `src/e12_transformer.py`, ktorý pre každý zo štyroch štítkov doladí samostatný model na title + text a po každej epoche vyberie najlepšiu epochu **len podľa valid**.

- **E1** = bez váh tried, **E2** = s váhami tried (`--vahy`); každý variant so semienkami 42, 43 a 44.
- Výstupy idú do `results/etapa2/` v dohodnutom formáte (predikcie, top-10 kandidátov, log), takže ich ďalšie skripty (ensemble, kombinácia, LLM, vyhodnotenie) použijú priamo.
- Test sa tu **nevyhodnocuje**. Predikcie na test sa len uložia, vyhodnotí ich osoba 1 jediný raz na konci.
- Nameraný čas na Tesla T4: jeden beh E1 alebo E2 asi 22 minút, jeden beh E1b asi 25 minút. Návod je v `docs/etapa2/navod_gpu.md`.

In [ ]:
# 1. stiahnutie repozitára (vetva 2. etapy) a inštalácia
!git clone -b etapa2-priprava https://github.com/alzbetajaksova-ux/NLP_food_hazard.git
%cd NLP_food_hazard
!pip install -q transformers==5.23.0

In [ ]:
# 2. kontrola GPU
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 3. nastavenia
MODEL = "roberta-base"      # prípadne "microsoft/deberta-v3-base" (zapíš do správy, ktorý si použila)
SEMIENKA = [42, 43, 44]
EPOCHY = 4

In [ ]:
# 3b. (odporúčané na Colabe) priebežné ukladanie na Google Drive, aby sa výsledky nestratili pri odpojení
ZALOHA = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ZALOHA = "/content/drive/MyDrive/nlp_food_hazard_etapa2"   # sem sa kopíruje results/etapa2 po každom behu
except ImportError:
    print("Nie je Colab (napr. Kaggle): zálohu rieš stiahnutím výstupov z /kaggle/working.")

In [ ]:
# 4. rýchla skúška na 200 správach (nič neukladá), overí, že všetko beží
!python src/e12_transformer.py --model {MODEL} --vzorka 200 --epochy 1

In [ ]:
# 5. celá mriežka: E1 a E2 × 3 semienka (dlhý beh)
# Behy, ktoré už majú log, sa preskočia, takže po odpojení stačí bunky spustiť znova (vrátane 3b).
import os, shutil, subprocess, sys
if ZALOHA and os.path.exists(ZALOHA):                       # obnovenie z Drive po odpojení
    shutil.copytree(ZALOHA, "results/etapa2", dirs_exist_ok=True)
kratky = os.path.basename(MODEL.rstrip("/")).replace("-", "").replace("_", "")
for vahy in ["", "--vahy"]:
    for seed in SEMIENKA:
        system = f"{'E2' if vahy else 'E1'}_{kratky}_s{seed}"
        if os.path.exists(f"results/etapa2/logy/{system}.json"):
            print("už hotové:", system)
            continue
        prikaz = f"{sys.executable} src/e12_transformer.py --model {MODEL} --seed {seed} --epochy {EPOCHY} {vahy}"
        print(">>>", prikaz, flush=True)
        subprocess.run(prikaz.split(), check=True)
        if ZALOHA:
            shutil.copytree("results/etapa2", ZALOHA, dirs_exist_ok=True)
            print("zálohované na Drive:", system)

In [ ]:
# 6. porovnanie na valid: priemer a smerodajná odchýlka cez semienka
import sys, glob, os
import pandas as pd
sys.path.insert(0, "src")
from spolocne import nacitaj, nacitaj_predikcie, skore

valid = nacitaj("valid")
riadky = []
for cesta in sorted(glob.glob("results/etapa2/predikcie/E[12]_*_valid.csv")):
    system = os.path.basename(cesta).removesuffix("_valid.csv")
    s = skore(valid, nacitaj_predikcie(cesta))
    riadky.append({"systém": system, "variant": system.split("_s")[0], **{k: round(v, 4) for k, v in s.items()}})
tab = pd.DataFrame(riadky)
display(tab)
display(tab.drop(columns="systém").groupby("variant").agg(["mean", "std"]).round(4))
e0 = skore(valid, nacitaj_predikcie("results/etapa2/predikcie/E0_svm_valid.csv"))
print(f"pre porovnanie E0 (SVM) na valid: ST1 {e0['ST1']:.4f}, ST2 {e0['ST2']:.4f}")

In [ ]:
# 7. párový bootstrap jedného semienka proti E0 (na valid)
!python src/vyhodnot.py --split valid E2_robertabase_s42 --porovnaj E0_svm --bootstrap 1000 --kandidati E2_robertabase_s42

## Doplnkový experiment E1b: dlhší tréning pre hazard a product

Prvá mriežka (15. 11.) ukázala, že pri 4 epochách a lr 2e-5 sa transformer nestihne naučiť jemné štítky s mnohými triedami: najlepšia epocha bola takmer vždy posledná a pri produkte (1 022 tried) mal macro-F1 0,08 (E1) a 0,00 (E2 s váhami). Tento experiment dolaďuje len `hazard` a `product`, bez váh tried, 10 epoch a s vyšším lr pre klasifikačnú hlavu. Ostatné štítky sa doplnia z `E0_svmvyber`. Nameraný čas: asi 25 minút na semienko.

In [ ]:
# 9. E1b: hazard a product, 10 epoch, lr hlavy 1e-3, bez váh; 3 semienka (preskočí hotové)
import os, shutil, subprocess, sys
if ZALOHA and os.path.exists(ZALOHA):
    shutil.copytree(ZALOHA, "results/etapa2", dirs_exist_ok=True)
for seed in SEMIENKA:
    system = f"E1b_robertabase_s{seed}"
    if os.path.exists(f"results/etapa2/logy/{system}.json"):
        print("už hotové:", system); continue
    prikaz = (f"{sys.executable} src/e12_transformer.py --model {MODEL} --seed {seed} --epochy 10 "
              f"--lr 3e-5 --lr-hlava 1e-3 --stitky hazard product --doplnit-z E0_svmvyber --system {system}")
    print(">>>", prikaz, flush=True)
    subprocess.run(prikaz.split(), check=True)
    if ZALOHA:
        shutil.copytree("results/etapa2", ZALOHA, dirs_exist_ok=True)
        print("zálohované na Drive:", system)
!python src/vyhodnot.py --split valid E0_svmvyber E1b_robertabase_s42 E1b_robertabase_s43 E1b_robertabase_s44 --kandidati E1b_robertabase_s42

## Ako odovzdať výsledky

1. Stiahni `vysledky_transformer.zip` (bunka nižšie) a nahraj jeho obsah do repozitára do `results/etapa2/` vo vetve `etapa2-priprava` (alebo commitni priamo z Colabu).
2. Do správy zapíš: model, počet epoch, lr, batch, max. dĺžku, typ GPU a časy z `results/etapa2/logy/*.json`.
3. Potom sa spustí `python src/zostav_systemy.py`, ktorý z behov zostaví ensemble a výsledné systémy; osoba 2 použije kandidátov `E12kens` pre LLM.
4. Použitie AI pri práci zapíš do prílohy (nástroj, prompt, výstup).

In [ ]:
# 8. zabalenie výsledkov na stiahnutie
!zip -qr vysledky_transformer.zip results/etapa2/predikcie results/etapa2/kandidati results/etapa2/logy
from google.colab import files
files.download("vysledky_transformer.zip")